In [ ]:
# Step 1: Install dependencies
!pip install pandas openpyxl sentence-transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
df_train = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/Tucker/FineTuning.xlsx")
df_comments = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/Tucker/cleaned_comments.xlsx")


In [ ]:
# Define target columns
target_cols = [
    'america_first_score',
    'israel_first_score',
    'aipac_influence_score',
    'palestine_score'
]

In [ ]:
X_text_train = df_train['text'].tolist()
y_train = df_train[target_cols].values

In [ ]:
# Step 3: Load Hugging Face Pre-trained Model
# 'all-mpnet-base-v2' or 'all-MiniLM-L6-v2' are available on Hugging Face
model_name = "sentence-transformers/all-mpnet-base-v2"
embedding_model = SentenceTransformer(model_name)

# Extract embeddings
print("Encoding training samples into vectors...")
X_embeddings_train = embedding_model.encode(X_text_train, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding training samples into vectors...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Step 4: Train Multi-Output Regressor
regressor = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
regressor.fit(X_embeddings_train, y_train)

MultiOutputRegressor(estimator=RandomForestRegressor(random_state=42))

In [ ]:
# Step 5: Inference on Full Dataset (55,841 comments)
print("Encoding full comment dataset...")
X_embeddings_all = embedding_model.encode(df_comments['text'].tolist(), batch_size=64, show_progress_bar=True)

Encoding full comment dataset...


Batches:   0%|          | 0/873 [00:00<?, ?it/s]

In [ ]:
# Predict scores
predictions = regressor.predict(X_embeddings_all)

# Save results back to DataFrame
for i, col in enumerate(target_cols):
    df_comments[col] = predictions[:, i]

In [ ]:
output_path = "/content/drive/MyDrive/Colab Notebooks/Tucker/scored_cleaned_comments.xlsx"
df_comments.to_excel(output_path, index=False)
print(f"Done! Results saved to {output_path}")

Done! Results saved to /content/drive/MyDrive/Colab Notebooks/Tucker/scored_cleaned_comments.xlsx
